In [ ]:
# =========================
# 2. Configuration
# =========================
from pathlib import Path
import os
import json
import math
import random
import warnings

BASE_DIR = Path(".")

IMAGE_DIR = BASE_DIR / "data" / "images" / "val"
SALIENCY_DIR = BASE_DIR / "data" / "maps" / "val"
COCO_ANNOTATION_FILE = BASE_DIR / "data" / "annotations" / "captions_val2014.json"

OUTPUT_DIR = BASE_DIR / "outputs" / "vitgpt2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional audited manual labels:
MANUAL_LABELS_FILE = BASE_DIR / "data" / "annotations" / "manual_labels.csv"

MODEL_NAME = "nlpconnect/vit-gpt2-image-captioning"

# Set to an integer for a smoke test; set to None for the complete dataset.
MAX_IMAGES = 10

SEED = 42
DEVICE_PREFERENCE = "cuda"

# Save per-image token-level records and diagnostic metadata.
SAVE_TOKEN_RECORDS = True
SAVE_JSONL = True

print("IMAGE_DIR:", IMAGE_DIR)
print("SALIENCY_DIR:", SALIENCY_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# =========================
# 3. Reproducibility
# =========================
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    DEVICE_PREFERENCE if DEVICE_PREFERENCE == "cuda" and torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)
print("Torch:", torch.__version__)


In [ ]:
# =========================
# 4. Imports
# =========================
import pandas as pd
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

from transformers import (
    VisionEncoderDecoderModel,
    ViTImageProcessor,
    AutoTokenizer,
)

print("Imports successful.")


In [ ]:
# =========================
# 5. Load audited baseline model
# =========================
model = VisionEncoderDecoderModel.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager",
)

image_processor = ViTImageProcessor.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# We need decoder cross-attentions from an explicit forward pass.
model.config.output_attentions = True
if hasattr(model.config, "decoder"):
    model.config.decoder.output_attentions = True

model.to(device)
model.eval()

print("Loaded:", MODEL_NAME)
print("Decoder start token:", model.config.decoder_start_token_id)
print("EOS token:", model.config.eos_token_id)
print("Tokenizer vocab size:", len(tokenizer))


In [ ]:
# =========================
# 7. Numerically stable distribution utilities
# =========================
def normalize_distribution(x, eps=1e-12):
    """Convert a non-negative spatial map to a probability distribution."""
    x = np.asarray(x, dtype=np.float64)

    if x.size == 0:
        raise ValueError("Cannot normalize an empty array.")

    if not np.all(np.isfinite(x)):
        raise ValueError("Distribution contains NaN/Inf values.")

    # Attention should be non-negative. Clamp tiny numerical negatives.
    x = np.maximum(x, 0.0)

    total = x.sum()
    if total <= eps:
        # Uniform fallback is explicit rather than producing NaNs.
        return np.full(x.shape, 1.0 / x.size, dtype=np.float64)

    return x / total


def kl_divergence(p, q, eps=1e-12):
    p = normalize_distribution(p, eps)
    q = normalize_distribution(q, eps)
    return float(np.sum(p * np.log((p + eps) / (q + eps))))


def js_divergence(p, q, eps=1e-12):
    p = normalize_distribution(p, eps)
    q = normalize_distribution(q, eps)
    m = 0.5 * (p + q)
    return float(
        0.5 * kl_divergence(p, m, eps)
        + 0.5 * kl_divergence(q, m, eps)
    )


def entropy(p, eps=1e-12):
    p = normalize_distribution(p, eps)
    return float(-np.sum(p * np.log(p + eps)))


In [ ]:
# =========================
# 8. Deterministic image/saliency pairing
# =========================
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

image_files = sorted(
    p for p in IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

if MAX_IMAGES is not None:
    image_files = image_files[:MAX_IMAGES]

pairs = []
missing_saliency = []

for image_path in image_files:
    saliency_path = SALIENCY_DIR / f"{image_path.stem}.png"
    if saliency_path.exists():
        pairs.append((image_path, saliency_path))
    else:
        missing_saliency.append({
            "image_id": image_path.stem,
            "image_path": str(image_path),
            "expected_saliency_path": str(saliency_path),
        })

print("Candidate images:", len(image_files))
print("Valid image/saliency pairs:", len(pairs))
print("Missing saliency maps:", len(missing_saliency))

if missing_saliency:
    pd.DataFrame(missing_saliency).head()


In [ ]:
# =========================
# 9. Load + validate saliency
# =========================
def load_saliency(saliency_path, grid_size):
    """Load grayscale saliency and resize it to the encoder spatial grid."""
    with Image.open(saliency_path) as im:
        saliency = im.convert("L")
        saliency = saliency.resize(
            (grid_size, grid_size),
            Image.Resampling.BILINEAR
        )

    saliency_np = np.asarray(saliency, dtype=np.float64)

    if saliency_np.ndim != 2:
        raise ValueError(f"Expected 2-D grayscale saliency, got {saliency_np.shape}")

    return normalize_distribution(saliency_np)


In [ ]:
# =========================
# 10. Attention extraction helpers
# =========================
def infer_spatial_grid(num_encoder_tokens):
    """Return spatial grid size after removing CLS.

    ViT commonly exposes 1 CLS token + N spatial patch tokens.
    We validate that N is a perfect square instead of hard-coding 14x14.
    """
    if num_encoder_tokens < 2:
        raise ValueError(
            f"Encoder sequence is too short ({num_encoder_tokens}) to contain CLS + patches."
        )

    num_patches = num_encoder_tokens - 1
    grid_size = int(math.isqrt(num_patches))

    if grid_size * grid_size != num_patches:
        raise ValueError(
            f"After CLS removal, {num_patches} encoder tokens do not form a square grid."
        )

    return grid_size, num_patches


def extract_last_layer_cross_attention(forward_outputs):
    """Extract [tokens, spatial_patches] from final decoder layer.

    Returns attention after:
      batch removal -> head mean -> CLS removal.
    """
    cross_attentions = forward_outputs.cross_attentions

    if cross_attentions is None or len(cross_attentions) == 0:
        raise RuntimeError(
            "Cross-attentions were not returned. Ensure eager attention and "
            "output_attentions=True are enabled."
        )

    last = cross_attentions[-1]

    if last.ndim != 4:
        raise ValueError(f"Unexpected cross-attention rank: {last.shape}")

    # [batch, heads, decoder_tokens, encoder_tokens]
    if last.shape[0] != 1:
        raise ValueError("This notebook currently expects batch size 1.")

    last = last[0]
    attention = last.mean(dim=0)  # [decoder_tokens, encoder_tokens]

    grid_size, num_patches = infer_spatial_grid(attention.shape[-1])

    # Remove CLS from encoder-side spatial tokens.
    attention = attention[:, 1:]

    if attention.shape[-1] != num_patches:
        raise AssertionError("CLS removal produced an unexpected patch count.")

    token_maps = []
    for row in attention:
        # Softmax attention is non-negative, but normalize defensively.
        spatial = row.detach().float().cpu().numpy()
        spatial = normalize_distribution(spatial)
        token_maps.append(spatial.reshape(grid_size, grid_size))

    token_maps = np.stack(token_maps, axis=0)

    return token_maps, grid_size, cross_attentions


In [ ]:
# =========================
# 12. Token metadata + alignment
# =========================
def build_token_records(generated_ids, caption, token_maps, tokenizer):
    """Build auditable token records aligned to target sequence positions."""
    ids = generated_ids[0].detach().cpu().tolist()
    tokens = tokenizer.convert_ids_to_tokens(ids)

    special_ids = set(
        x for x in [
            tokenizer.bos_token_id,
            tokenizer.eos_token_id,
            tokenizer.pad_token_id,
            tokenizer.cls_token_id,
            tokenizer.sep_token_id,
        ]
        if x is not None
    )

    records = []

    # token_maps has one row per target sequence position from the forward pass.
    n = min(len(ids), token_maps.shape[0])

    for pos in range(n):
        token_id = ids[pos]
        token = tokens[pos]

        is_special = token_id in special_ids or token in tokenizer.all_special_tokens

        rec = {
            "sequence_position": pos,
            "token_id": int(token_id),
            "token": token,
            "is_special": bool(is_special),
            "is_semantic_token": bool(not is_special),
            "attention_entropy": entropy(token_maps[pos]),
            "attention_map": token_maps[pos].tolist(),
        }

        records.append(rec)

    return records


In [ ]:
# =========================
# 13. Single-image audited inference
# =========================
def process_image_vit_gpt2(image_path, saliency_path):
    image_path = Path(image_path)
    saliency_path = Path(saliency_path)

    with Image.open(image_path) as image:
        image = image.convert("RGB")

        pixel_values = image_processor(
            images=image,
            return_tensors="pt"
        ).pixel_values.to(device)

    # Generation: deterministic decoding for reproducibility.
    with torch.no_grad():
        generated = model.generate(
            pixel_values=pixel_values,
            do_sample=False,
            return_dict_in_generate=True,
        )

    generated_ids = generated.sequences
    caption = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    ).strip()

    # Forward pass with labels gives standard decoder-input shifting.
    with torch.no_grad():
        forward_outputs = model(
            pixel_values=pixel_values,
            labels=generated_ids,
            output_attentions=True,
            return_dict=True,
        )

    token_maps, grid_size, cross_attentions = (
        extract_last_layer_cross_attention(forward_outputs)
    )

    saliency_np = load_saliency(saliency_path, grid_size)

    token_records = build_token_records(
        generated_ids,
        caption,
        token_maps,
        tokenizer,
    )

    # Semantic-token diagnostics only.
    semantic_records = [
        r for r in token_records
        if r["is_semantic_token"]
        and r["sequence_position"] < token_maps.shape[0]
    ]

    token_js = []
    token_kl = []

    for r in semantic_records:
        amap = np.asarray(r["attention_map"], dtype=np.float64)
        js = js_divergence(amap, saliency_np)
        kl = kl_divergence(amap, saliency_np)

        r["JS"] = js
        r["KL"] = kl

        token_js.append(js)
        token_kl.append(kl)

    return {
        "image_id": image_path.stem,
        "caption": caption,
        "num_sequence_tokens": int(len(token_records)),
        "num_semantic_tokens": int(len(semantic_records)),
        "grid_size": int(grid_size),
        "avg_JS": float(np.mean(token_js)) if token_js else np.nan,
        "avg_KL": float(np.mean(token_kl)) if token_kl else np.nan,
        "min_JS": float(np.min(token_js)) if token_js else np.nan,
        "max_JS": float(np.max(token_js)) if token_js else np.nan,
        "min_KL": float(np.min(token_kl)) if token_kl else np.nan,
        "max_KL": float(np.max(token_kl)) if token_kl else np.nan,
        "token_records": semantic_records if SAVE_TOKEN_RECORDS else None,
        "saliency_map": saliency_np.tolist() if SAVE_TOKEN_RECORDS else None,
        "model_name": MODEL_NAME,
        "attention_layer": "decoder_last",
        "attention_head_reduction": "mean",
        "decoding": "greedy/deterministic",
    }


In [ ]:
# =========================
# 14. Smoke test before full run
# =========================
if not pairs:
    raise FileNotFoundError(
        "No image/saliency pairs found. Update IMAGE_DIR and SALIENCY_DIR."
    )

test_image, test_saliency = pairs[0]
test_result = process_image_vit_gpt2(test_image, test_saliency)

print("Image:", test_result["image_id"])
print("Caption:", test_result["caption"])
print("Grid:", test_result["grid_size"], "x", test_result["grid_size"])
print("Semantic tokens:", test_result["num_semantic_tokens"])
print("Average JS:", test_result["avg_JS"])
print("Average KL:", test_result["avg_KL"])


In [ ]:
# =========================
# 15. Structural sanity checks
# =========================
assert test_result["grid_size"] > 0
assert np.isfinite(test_result["avg_JS"])
assert np.isfinite(test_result["avg_KL"])

for record in test_result["token_records"]:
    amap = np.asarray(record["attention_map"], dtype=np.float64)
    assert amap.shape == (
        test_result["grid_size"],
        test_result["grid_size"]
    )
    assert np.isfinite(amap).all()
    assert np.isclose(amap.sum(), 1.0, atol=1e-6)
    assert 0.0 <= record["JS"]
    assert record["KL"] >= 0.0

sal = np.asarray(test_result["saliency_map"], dtype=np.float64)
assert sal.shape == (
    test_result["grid_size"],
    test_result["grid_size"]
)
assert np.isclose(sal.sum(), 1.0, atol=1e-6)

print("All smoke-test structural checks passed.")


In [ ]:
# =========================
# 17. Run dataset analysis
# =========================
results = []
failures = []

for image_path, saliency_path in tqdm(pairs, desc="ViT-GPT2 analysis"):
    try:
        result = process_image_vit_gpt2(image_path, saliency_path)
        results.append(result)
    except (OSError, UnidentifiedImageError, ValueError, RuntimeError) as exc:
        failures.append({
            "image_id": image_path.stem,
            "image_path": str(image_path),
            "saliency_path": str(saliency_path),
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        })

print("Successful:", len(results))
print("Failed:", len(failures))


In [ ]:
# =========================
# 18. Flatten summary output
# =========================
summary_rows = []

for r in results:
    summary_rows.append({
        "image_id": r["image_id"],
        "caption": r["caption"],
        "num_sequence_tokens": r["num_sequence_tokens"],
        "num_semantic_tokens": r["num_semantic_tokens"],
        "grid_size": r["grid_size"],
        "avg_JS": r["avg_JS"],
        "avg_KL": r["avg_KL"],
        "min_JS": r["min_JS"],
        "max_JS": r["max_JS"],
        "min_KL": r["min_KL"],
        "max_KL": r["max_KL"],
        "model_name": r["model_name"],
        "attention_layer": r["attention_layer"],
        "attention_head_reduction": r["attention_head_reduction"],
        "decoding": r["decoding"],
    })

df = pd.DataFrame(summary_rows)

if not df.empty:
    print(df.head())
    print("Processed images:", len(df))
else:
    print("No successful results.")


In [ ]:
# =========================
# 19. Save summary + failures
# =========================
summary_path = OUTPUT_DIR / "vitgpt2_attention_alignment_results.csv"
failure_path = OUTPUT_DIR / "vitgpt2_attention_failures.csv"

df.to_csv(summary_path, index=False)
pd.DataFrame(failures).to_csv(failure_path, index=False)

print("Saved:", summary_path)
print("Saved:", failure_path)


In [ ]:
# =========================
# 20. Save token-level JSONL
# =========================
if SAVE_JSONL:
    jsonl_path = OUTPUT_DIR / "vitgpt2_token_attention_records.jsonl"

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for result in results:
            record = {
                "image_id": result["image_id"],
                "caption": result["caption"],
                "grid_size": result["grid_size"],
                "avg_JS": result["avg_JS"],
                "avg_KL": result["avg_KL"],
                "model_name": result["model_name"],
                "attention_layer": result["attention_layer"],
                "token_records": result["token_records"],
                "saliency_map": result["saliency_map"],
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print("Saved:", jsonl_path)


In [ ]:
# =========================
# 22. Optional COCO references
# =========================
gt_dict = {}

if COCO_ANNOTATION_FILE.exists():
    with open(COCO_ANNOTATION_FILE, "r", encoding="utf-8") as f:
        coco_data = json.load(f)

    for ann in coco_data.get("annotations", []):
        image_id = int(ann["image_id"])
        gt_dict.setdefault(image_id, []).append(ann["caption"])

    print("Loaded reference captions for", len(gt_dict), "images.")
else:
    print("COCO annotation file not found; skipping reference-caption loading.")


In [ ]:
# =========================
# 24. Merge audited manual labels
# =========================
if MANUAL_LABELS_FILE.exists() and not df.empty:
    labels = pd.read_csv(MANUAL_LABELS_FILE)

    required = {"image_id", "error_type"}
    missing = required - set(labels.columns)

    if missing:
        raise ValueError(f"Manual label file missing columns: {sorted(missing)}")

    labels["image_id"] = (
        labels["image_id"]
        .astype(str)
        .str.replace(".jpg", "", regex=False)
        .str.replace(".jpeg", "", regex=False)
        .str.replace(".png", "", regex=False)
        .str.strip()
    )

    df["image_id"] = df["image_id"].astype(str).str.strip()

    df_labeled = df.merge(
        labels,
        on="image_id",
        how="inner",
        suffixes=("", "_label")
    )

    print("Attention results:", len(df))
    print("Labeled matched results:", len(df_labeled))
else:
    df_labeled = pd.DataFrame()
    print("Manual labels unavailable or no results; skipping merge.")


In [ ]:
# =========================
# 25. Descriptive reliability analysis
# =========================
if not df_labeled.empty:
    df_labeled["has_error"] = df_labeled["error_type"] != 0

    print(
        df_labeled.groupby("has_error")[["avg_JS", "avg_KL"]]
        .agg(["count", "mean", "median", "std"])
    )
else:
    print("No labeled rows available.")


In [ ]:
# =========================
# 27. Tokenization inspection helper
# =========================
def inspect_caption_tokens(caption):
    encoded = tokenizer(
        caption,
        add_special_tokens=True,
        return_attention_mask=False,
    )
    ids = encoded["input_ids"]
    toks = tokenizer.convert_ids_to_tokens(ids)

    return pd.DataFrame({
        "position": range(len(ids)),
        "token_id": ids,
        "token": toks,
        "special": [
            tok in tokenizer.all_special_tokens
            for tok in toks
        ],
    })

if results:
    print(results[0]["caption"])
    display(inspect_caption_tokens(results[0]["caption"]))


In [ ]:
# =========================
# 29. Visualization helper
# =========================
import matplotlib.pyplot as plt


def get_result(image_id):
    for r in results:
        if r["image_id"] == image_id:
            return r
    raise KeyError(f"Result not found: {image_id}")


def plot_token_attention(image_id, token_index=0):
    r = get_result(image_id)

    semantic = r["token_records"]
    if not semantic:
        raise ValueError("No semantic tokens available.")

    if token_index < 0 or token_index >= len(semantic):
        raise IndexError("token_index outside semantic-token range.")

    record = semantic[token_index]

    image_path = next(
        p for p, _ in pairs
        if p.stem == image_id
    )

    with Image.open(image_path) as im:
        image = im.convert("RGB")

    attn = np.asarray(record["attention_map"], dtype=np.float64)
    sal = np.asarray(r["saliency_map"], dtype=np.float64)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    axes[0].imshow(image)
    axes[0].set_title(f"Image\n{r['caption']}")
    axes[0].axis("off")

    axes[1].imshow(attn, interpolation="nearest")
    axes[1].set_title(
        f"Attention: {record['token']}\nJS={record['JS']:.4f}"
    )
    axes[1].axis("off")

    axes[2].imshow(sal, interpolation="nearest")
    axes[2].set_title("Saliency")
    axes[2].axis("off")

    axes[3].imshow(image)
    axes[3].imshow(
        Image.fromarray(
            np.uint8(255 * (attn - attn.min()) / (attn.max() - attn.min() + 1e-12))
        ).resize(image.size),
        alpha=0.45,
        cmap="magma",
    )
    axes[3].set_title("Attention overlay")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# =========================
# 30. Example visualization
# =========================
if results and results[0]["token_records"]:
    plot_token_attention(results[0]["image_id"], token_index=0)


In [ ]:
# =========================
# 32. Optional Welch t-test
# =========================
from scipy.stats import ttest_ind

if not df_labeled.empty:
    correct = df_labeled.loc[~df_labeled["has_error"], "avg_JS"].dropna()
    error = df_labeled.loc[df_labeled["has_error"], "avg_JS"].dropna()

    if len(correct) >= 2 and len(error) >= 2:
        test = ttest_ind(correct, error, equal_var=False)
        print("Welch t-test for avg_JS")
        print("Correct n:", len(correct))
        print("Error n:", len(error))
        print("Statistic:", test.statistic)
        print("p-value:", test.pvalue)
    else:
        print("Not enough observations for Welch t-test.")
else:
    print("No labeled data.")


In [ ]:
# =========================
# 34. Final output manifest
# =========================
manifest = {
    "model_name": MODEL_NAME,
    "device": str(device),
    "seed": SEED,
    "image_dir": str(IMAGE_DIR),
    "saliency_dir": str(SALIENCY_DIR),
    "num_candidate_images": len(image_files),
    "num_pairs": len(pairs),
    "num_successful": len(results),
    "num_failed": len(failures),
    "attention_source": "final decoder cross-attention",
    "head_reduction": "mean",
    "encoder_cls_removed": True,
    "spatial_grid": (
        int(results[0]["grid_size"]) if results else None
    ),
    "token_alignment": "forward(labels=generated_ids), special tokens excluded from semantic analysis",
    "decoding": "greedy/deterministic",
    "bleu_as_correctness_label": False,
    "notes": "Audited Major baseline for token-level visual attention/saliency analysis.",
}

manifest_path = OUTPUT_DIR / "vitgpt2_analysis_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print("Manifest:", manifest_path)
